# Weight-averaged control of the contested towns on Blotto (appendix figure)
Contested towns are those both valued and reachable by the attacker. Two families of 100 episodes each place the valued towns among the reachable ones: one valued town with weight 1, or two valued towns with weight 0.5 each. For each defender, the figure reports the mean over episodes of the weight-averaged control of the contested towns, against the trained attacker. Every defender samples its action from its own distribution.

In [ ]:
%run Initialize.ipynb

## Settings and trained models

In [ ]:
SAVE_DIR = "compared"

Context_Model = Load(TSTransformer().to(device), "encoder.pth")
Z_S_ENC       = Load(StateGRU().to(device), "z_s_enc.pth")
critic        = Load(Critic().to(device), "critic.pth")
Player1       = Load(Player1Actor().to(device), "player1.pth")
BASELINES     = Load_Baselines(SAVE_DIR)
Player2       = Load(Player2Actor().to(device), "player2.pth")

CASE_WEIGHTS = {'1 valued': [1.0], '2 valued': [0.5, 0.5]}
alloc_episodes = 100


def make_case_configs(w_vals, n_eps, seed):
    rng = np.random.RandomState(seed)
    cfgs = []
    for _ in range(n_eps):
        Theta = np.zeros(Len_Theta)
        Theta[0:N] = rng.choice(RHO_VALUES, size=N)
        Reachable = np.zeros(N)
        reach_idx = rng.choice(N, size=3, replace=False)
        Reachable[reach_idx] = 1.0
        valued = rng.choice(reach_idx, size=len(w_vals), replace=False)
        Weights = np.zeros(N)
        Weights[valued] = w_vals
        Type_Theta = np.concatenate([Weights, Reachable])
        Theta[N:Len_Theta] = Type_Theta
        x0 = rng.beta(2.0, 2.0, size=N)
        cfgs.append((Theta.copy(), Type_Theta.copy(), Weights.copy(), x0.copy()))
    return cfgs

## Rollout

In [ ]:
def contested_control(Defender, cfgs):
    defw = get_defender(Defender)
    ep_means = []
    for i_episode, (Theta, Type_Theta, Weights, x0) in enumerate(tqdm(cfgs, desc=Defender, leave=False)):
        np.random.seed(2000 + i_episode); torch.manual_seed(2000 + i_episode); random.seed(2000 + i_episode)
        Reachable = Theta[2*N:3*N]
        contested = (Weights > 0) & (Reachable > 0)
        w_norm = Weights[contested] / Weights[contested].sum()
        legal = legal_actions(Reachable)
        defw.reset(Theta, Type_Theta, legal)
        tt_t = torch.tensor(Type_Theta, dtype=torch.float32, device=device)
        x = x0.copy()
        m2 = np.full(N * 2, 0.5)
        ctrl = []
        for t in range(max_steps):
            a_dist = defw.act_dist(x)
            with torch.no_grad():
                u_dist = Player2(torch.tensor(x, dtype=torch.float32, device=device).unsqueeze(0),
                                 torch.tensor(m2, dtype=torch.float32, device=device).unsqueeze(0),
                                 tt_t.unsqueeze(0)).squeeze(0).cpu().numpy()
            u = int(np.random.choice(n_u, p=u_dist / u_dist.sum()))
            a = int(np.random.choice(n_a, p=a_dist))
            next_x = next_state(x, actions[a], actions[u], Theta)
            ctrl.append(float((w_norm * next_x[contested]).sum()))
            next_m2 = m2_update(m2, x, actions[u], next_x)
            y = Observation(next_x)
            defw.observe(x, actions[a], actions[u], next_x, y)
            x, m2 = next_x, next_m2
        ep_means.append(np.mean(ctrl))
    return np.asarray(ep_means)


PLOT_DEFENDERS = ['Critic P1', 'Trained P1', 'RPPO', 'DR_RPPO', 'VARIBAD', 'PEARL', 'MMD']
case_names = list(CASE_WEIGHTS.keys())
case_cfgs = {c: make_case_configs(CASE_WEIGHTS[c], alloc_episodes, seed=2026 + i) for i, c in enumerate(case_names)}
alloc_results = {}
for c in case_names:
    for d_ in PLOT_DEFENDERS:
        m = contested_control(d_, case_cfgs[c])
        alloc_results[(c, d_)] = (m.mean(), m.std(ddof=1) / np.sqrt(len(m)))
        print(f"{c:<12}{d_:<14}{m.mean():.3f}+/-{m.std(ddof=1) / np.sqrt(len(m)):.3f}")

## Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
xs = np.arange(len(case_names))
xw = 0.8 / len(PLOT_DEFENDERS)
for j, d_ in enumerate(PLOT_DEFENDERS):
    means = [alloc_results[(c, d_)][0] for c in case_names]
    ses = [alloc_results[(c, d_)][1] for c in case_names]
    ax.bar(xs + (j - (len(PLOT_DEFENDERS) - 1) / 2) * xw, means, width=xw * 0.9, yerr=ses, capsize=2, label=d_)
ax.set_xticks(xs); ax.set_xticklabels(case_names, fontsize=14)
ax.set_ylabel('Weight-averaged control (contested towns)', fontsize=12)
ax.set_ylim(0, 0.6)
plt.yticks([0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6], fontsize=12)
ax.legend(ncol=4, fontsize=10, loc='upper center')
ax.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('appendix_control.jpg', dpi=300, bbox_inches='tight')
plt.show()